In [1]:
import pandas as pd
import numpy as np
import time
import csv
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.seasonal import STL
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, median_absolute_error
from pmdarima.arima import auto_arima, ADFTest, ndiffs
from pmdarima.arima import StepwiseContext
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import os

# ==========================================
# 1. CONFIGURATIONS
# ==========================================

file_data = 'SN_m_tot_V2.0.csv'
path_name = '../datasets/'
path_name_results = '../results/'
path_name_figures = '../figures/'
file_result = 'Result_STLASL_sunspot.csv'

SHOW_PLOTS = False

# ==========================================
# 2. LOAD DATASET
# ==========================================

print("=" * 70)
print("STL-ARIMA-SVR-LSTM HYBRID MODEL")
print("Sunspot Numbers Forecasting")
print("=" * 70)

print("\n1. Loading Sunspot dataset...")
dataset_raw = pd.read_csv(f'{path_name}{file_data}', sep=';', encoding='latin1', decimal='.')
dataset_raw.columns = ['year', 'month', 'date', 'total_sunspot_number', 
                       'std_derivation', 'num_observations', 'def_prov_indicator']

dataset = pd.DataFrame()
dataset['date'] = pd.to_datetime(dataset_raw[['year', 'month']].assign(day=1))
dataset['num_observations'] = dataset_raw['num_observations'].values

print(f"Data loaded: {len(dataset)} records")
print(f"Period: {dataset['date'].iloc[0]} to {dataset['date'].iloc[-1]}")

# ==========================================
# 3. UTILITY FUNCTIONS
# ==========================================

def salvar_resultado(nm_dataset, ds_best_param, n_time_steps, MSE, RMSE, MAE, MAPE, sMAPE, Duration):
    data = [nm_dataset, ds_best_param, n_time_steps, MSE, RMSE, MAE, MAPE, sMAPE, Duration]
    fields = ['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
    
    os.makedirs(path_name_results, exist_ok=True)
    
    with open(f'{path_name_results}{file_result}', "a", newline='') as csv_file:
        writer = csv.writer(csv_file, delimiter=';')
        writer.writerow(data)
    print(fields)
    print(data)

def criar_arquivo_resultado():
    fields = ['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
    
    os.makedirs(path_name_results, exist_ok=True)
    
    with open(f'{path_name_results}{file_result}', "w", newline='') as csv_file:
        writer = csv.writer(csv_file, delimiter=';')
        writer.writerow(fields)

def create_lagged_features(dataset, n_time_steps):
    X, Y = [], []
    
    if n_time_steps == 0:
        for i in range(len(dataset) - 1):
            X.append([1])
            Y.append(dataset[i + 1])
    else:
        for i in range(len(dataset) - n_time_steps - 1):
            X.append(dataset[i:i + n_time_steps])
            Y.append(dataset[i + n_time_steps])
    
    return np.array(X), np.array(Y)

def calculate_metrics(y_test, predict):
    y_test = np.array(y_test).flatten()
    predict = np.array(predict).flatten()
    
    mse = mean_squared_error(y_test, predict)
    rmse = np.sqrt(mse)
    mae = median_absolute_error(y_pred=predict, y_true=y_test)
    mape = (np.mean(np.abs(y_test - predict) / (y_test + 1e-10))) * 100
    smape = round(np.mean(np.abs(predict - y_test) / ((np.abs(predict) + np.abs(y_test)) + 1e-10)) * 100, 2)
    
    return mse, rmse, mae, mape, smape

# ==========================================
# 4. PLOTTING FUNCTION
# ==========================================

def plot_stl_hybrid_results(dates, ts, nlinhas, trend, seasonal, residual, 
                            test_dates, trend_predict, seasonal_predict, residual_predict,
                            y_test_combined, combined_predict, smape, nm_dataset, n_time_steps):
    fig, axes = plt.subplots(4, 1, figsize=(14, 12))
    
    axes[0].plot(dates, ts.values, label='Original', color='blue')
    axes[0].axvline(x=dates[nlinhas], color='red', linestyle='--', label='Train/Test Split')
    axes[0].set_title(f'Original Time Series - {nm_dataset} (Sunspot Numbers)')
    axes[0].set_ylabel('Sunspot Number')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(dates, trend, label='Trend', color='green')
    axes[1].plot(dates, seasonal, label='Seasonal', color='orange')
    axes[1].plot(dates, residual, label='Residual', color='purple')
    axes[1].axvline(x=dates[nlinhas], color='red', linestyle='--')
    axes[1].set_title('STL Decomposition Components')
    axes[1].set_ylabel('Value')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(test_dates, trend_predict, label='Trend Prediction (ARIMA)', marker='o', markersize=3)
    axes[2].plot(test_dates, seasonal_predict, label='Seasonal Prediction (SVR)', marker='s', markersize=3)
    axes[2].plot(test_dates, residual_predict, label='Residual Prediction (LSTM)', marker='^', markersize=3)
    axes[2].set_title('Component Predictions')
    axes[2].set_ylabel('Value')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    axes[3].plot(test_dates, y_test_combined, label='Actual', color='blue', linewidth=2)
    axes[3].plot(test_dates, combined_predict, label='STL-ASL Prediction', 
                 color='red', linestyle='--', linewidth=2)
    axes[3].set_title(f'Final Prediction - sMAPE: {smape}%')
    axes[3].set_xlabel('Date')
    axes[3].set_ylabel('Sunspot Number')
    axes[3].legend()
    axes[3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    os.makedirs(path_name_figures, exist_ok=True)
    plt.savefig(f'{path_name_figures}stl_asl_{nm_dataset}_{n_time_steps}.pdf', 
                dpi=300, format='pdf', bbox_inches='tight')
    
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)

# ==========================================
# 5. INDIVIDUAL MODEL FUNCTIONS
# ==========================================

def previsao_ARIMA_component(data, n_time_steps, max_iter=500):
    if len(data) < n_time_steps + 10:
        return None, None, None, None
    
    try:
        nlinhas = int(len(data) * 0.80)
        train = data[:nlinhas]
        test = data[nlinhas:]
        
        X_train, Y_train = create_lagged_features(train, n_time_steps)
        X_test, Y_test = create_lagged_features(test, n_time_steps)
        
        if len(X_train) == 0 or len(X_test) == 0:
            return None, None, None, None
        
        Y_train = Y_train.reshape(-1, 1)
        
        adf_test = ADFTest(alpha=0.05)
        p_val, should_diff = adf_test.should_diff(Y_train)
        d = ndiffs(Y_train, test='adf') if should_diff else 0
        
        with StepwiseContext(max_dur=50):
            model = auto_arima(Y_train, X=X_train,
                               seasonal=True, m=12, maxiter=max_iter, d=d,
                               start_p=0, start_q=0, max_p=5, max_q=5,
                               D=None, stepwise=True, trace=False,
                               error_action='ignore', suppress_warnings=True)
        
        model.fit(Y_train)
        predict = model.predict(n_periods=len(Y_test), X=X_test)
        
        if hasattr(predict, 'shape') and len(predict.shape) > 1:
            predict = predict.flatten()
        
        return predict, Y_test.flatten(), model, str(model.order)
    
    except Exception as e:
        print(f"ARIMA component error: {e}")
        return None, None, None, None

def previsao_SVR_component(data, n_time_steps):
    if len(data) < n_time_steps + 10:
        return None, None, None, None
    
    try:
        nlinhas = int(len(data) * 0.80)
        train = data[:nlinhas]
        test = data[nlinhas:]
        
        X_train, Y_train = create_lagged_features(train, n_time_steps)
        X_test, Y_test = create_lagged_features(test, n_time_steps)
        
        if len(X_train) == 0 or len(X_test) == 0:
            return None, None, None, None
        
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        
        X_train_scaled = scaler_X.fit_transform(X_train)
        X_test_scaled = scaler_X.transform(X_test)
        Y_train_scaled = scaler_y.fit_transform(Y_train.reshape(-1, 1)).ravel()
        
        C = [12550, 125550, 1255555]
        gamma = [0.00001, 0.000001, 0.0000001, 0.00000001]
        epsilon = [0.1, 0.01, 0.001, 0.0001]
        
        hyper_params = [{'kernel': ['rbf'], 'C': C, 'gamma': gamma, 'epsilon': epsilon}]
        
        ts_cv = TimeSeriesSplit(n_splits=3, gap=2)
        
        grid = GridSearchCV(SVR(max_iter=1000), param_grid=hyper_params,
                            verbose=0, n_jobs=-1, cv=ts_cv,
                            scoring='neg_mean_absolute_percentage_error')
        
        grid.fit(X_train_scaled, Y_train_scaled)
        
        predict_scaled = grid.predict(X_test_scaled)
        predict = scaler_y.inverse_transform(predict_scaled.reshape(-1, 1)).ravel()
        
        return predict, Y_test.flatten(), grid, str(grid.best_params_)
    
    except Exception as e:
        print(f"SVR component error: {e}")
        return None, None, None, None

def previsao_LSTM_component(data, n_time_steps, l1=8, l2=18, l3=8, num_epochs=100, batch_size=32):
    if n_time_steps == 0:
        n_time_steps = 1
    
    if len(data) < n_time_steps + 10:
        return None, None, None, None
    
    try:
        data = np.array(data, dtype='float32')
        
        nlinhas = int(len(data) * 0.80)
        train = data[:nlinhas]
        test = data[nlinhas:]
        
        X_train, Y_train = create_lagged_features(train, n_time_steps)
        X_test, Y_test = create_lagged_features(test, n_time_steps)
        
        if len(X_train) == 0 or len(X_test) == 0:
            return None, None, None, None
        
        X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
        X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
        
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        
        X_train_flat = X_train.reshape(-1, 1)
        X_train_scaled = scaler_X.fit_transform(X_train_flat).reshape(X_train.shape)
        X_test_scaled = scaler_X.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)
        
        Y_train_scaled = scaler_y.fit_transform(Y_train.reshape(-1, 1)).ravel()
        
        model = Sequential()
        model.add(LSTM(l1, input_shape=(n_time_steps, 1), return_sequences=True))
        model.add(LSTM(l2, return_sequences=True))
        model.add(LSTM(l3))
        model.add(Dense(1))
        model.compile(loss='mean_squared_error', optimizer='adam')
        
        # Stops training when loss stops improving
        early_stop = EarlyStopping(
            monitor='loss',           # monitors training loss
            patience=20,              # waits 20 epochs before stopping
            restore_best_weights=True, # reverts to the best model found
            verbose=0,                # prints message when stopping
            min_delta=0.0001          # minimum change to qualify as improvement
        )

        # Train model with early_stop
        model.fit(
            X_train_scaled, Y_train_scaled,
            epochs=num_epochs,        
            batch_size=batch_size,
            verbose=0,                
            shuffle=False,            
            callbacks=[early_stop]
        )    
        
        predict_scaled = model.predict(X_test_scaled, batch_size=batch_size, verbose=0)
        predict = scaler_y.inverse_transform(predict_scaled).ravel()
        
        resultado = f"LSTM({l1},{l2},{l3})_epochs={num_epochs}"
        
        return predict, Y_test.flatten(), model, resultado
    
    except Exception as e:
        print(f"LSTM component error: {e}")
        return None, None, None, None

# ==========================================
# 6. MAIN HYBRID MODEL
# ==========================================

def previsao_STLASL(nm_dataset, dataset, n_time_steps, period=12):
    Hora_Inicio = time.time()
    
    data = dataset['num_observations'].values.astype('float64')
    dates = dataset['date'].values
    
    ts = pd.Series(data, index=dates, name='series')
    
    print(f"Applying STL decomposition (period={period})...")
    
    try:
        stl = STL(ts, period=period, robust=True)
        result = stl.fit()
        
        trend = result.trend.values
        seasonal = result.seasonal.values
        residual = result.resid.values
        
        trend = np.nan_to_num(trend)
        seasonal = np.nan_to_num(seasonal)
        residual = np.nan_to_num(residual)
        
    except Exception as e:
        print(f"STL decomposition failed: {e}")
        return None
    
    print("Predicting trend component with ARIMA...")
    trend_predict, trend_y_test, trend_model, trend_params = previsao_ARIMA_component(
        trend, n_time_steps, max_iter=2000
    )
    if trend_predict is None:
        print("Trend prediction failed")
        return None
    
    print("Predicting seasonal component with SVR...")
    seasonal_predict, seasonal_y_test, seasonal_model, seasonal_params = previsao_SVR_component(
        seasonal, n_time_steps
    )
    if seasonal_predict is None:
        print("Seasonal prediction failed")
        return None
    
    print("Predicting residual component with LSTM...")
    residual_predict, residual_y_test, residual_model, residual_params = previsao_LSTM_component(
        residual, n_time_steps, l1=8, l2=18, l3=8, num_epochs=200, batch_size=32
    )
    if residual_predict is None:
        print("Residual prediction failed")
        return None
    
    min_len = min(len(trend_predict), len(seasonal_predict), len(residual_predict))
    
    trend_predict = trend_predict[:min_len]
    seasonal_predict = seasonal_predict[:min_len]
    residual_predict = residual_predict[:min_len]
    
    nlinhas = int(len(data) * 0.80)
    
    combined_predict = trend_predict + seasonal_predict + residual_predict
    y_test_combined = data[nlinhas:nlinhas + min_len]
    
    mse, rmse, mae, mape, smape = calculate_metrics(y_test_combined, combined_predict)
    
    Hora_Fim = time.time()
    Duracao = Hora_Fim - Hora_Inicio
    
    resultado = f"STL(period={period})_ARIMA({trend_params})_SVR({seasonal_params})_LSTM({residual_params})"
    
    test_dates = dates[nlinhas:nlinhas + min_len]
    
    plot_stl_hybrid_results(
        dates=dates, ts=ts, nlinhas=nlinhas,
        trend=trend, seasonal=seasonal, residual=residual,
        test_dates=test_dates,
        trend_predict=trend_predict, seasonal_predict=seasonal_predict, residual_predict=residual_predict,
        y_test_combined=y_test_combined, combined_predict=combined_predict,
        smape=smape, nm_dataset=nm_dataset, n_time_steps=n_time_steps
    )
    
    salvar_resultado(nm_dataset, resultado, n_time_steps, mse, rmse, mae, mape, smape, Duracao)
    
    print(f"\nSTL-Hybrid Results for {nm_dataset} (n_time_steps={n_time_steps}):")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"sMAPE: {smape}%")
    print(f"Duration: {Duracao:.2f}s")
    
    return combined_predict

# ==========================================
# 7. MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    
    criar_arquivo_resultado()
    
    print("\n2. Testing different time windows (0 to 24 steps)...")
    print("=" * 70)
    
    for n_time_steps in range(0, 25):
        print(f"\n--- Processing n_time_steps={n_time_steps} ---")
        
        try:
            result = previsao_STLASL('sunspot', dataset, n_time_steps, period=12)
        except Exception as e:
            print(f"Error for n_time_steps={n_time_steps}: {e}")
            continue
    
    print("\n" + "=" * 70)
    print("Pipeline execution completed.")
    print(f"Results saved to: {path_name_results}{file_result}")
    print("=" * 70)

STL-ARIMA-SVR-LSTM HYBRID MODEL
Sunspot Numbers Forecasting

1. Loading Sunspot dataset...
Data loaded: 3295 records
Period: 1749-02-01 00:00:00 to 2023-08-01 00:00:00

2. Testing different time windows (0 to 24 steps)...

--- Processing n_time_steps=0 ---
Applying STL decomposition (period=12)...


Predicting trend component with ARIMA...


Predicting seasonal component with SVR...


Predicting residual component with LSTM...


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['sunspot', "STL(period=12)_ARIMA((0, 1, 0))_SVR({'C': 12550, 'epsilon': 0.1, 'gamma': 1e-05, 'kernel': 'rbf'})_LSTM(LSTM(8,18,8)_epochs=200)", 0, 336401.3770209036, 580.001187085771, 485.3690055245646, 73.8104132392999, 69.79, 58.96557021141052]

STL-Hybrid Results for sunspot (n_time_steps=0):
MSE: 336401.3770
RMSE: 580.0012
MAE: 485.3690
MAPE: 73.81%
sMAPE: 69.79%
Duration: 58.97s

--- Processing n_time_steps=1 ---
Applying STL decomposition (period=12)...


...

['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['sunspot', "STL(period=12)_ARIMA((0, 1, 1))_SVR({'C': 12550, 'epsilon': 0.01, 'gamma': 1e-07, 'kernel': 'rbf'})_LSTM(LSTM(8,18,8)_epochs=200)", 23, 292856.34676339634, 541.1620337416477, 480.0679978365332, 76.27281369875246, 68.2, 353.98841094970703]

STL-Hybrid Results for sunspot (n_time_steps=23):
MSE: 292856.3468
RMSE: 541.1620
MAE: 480.0680
MAPE: 76.27%
sMAPE: 68.2%
Duration: 353.99s

--- Processing n_time_steps=24 ---
Applying STL decomposition (period=12)...


Predicting trend component with ARIMA...


Predicting seasonal component with SVR...


Predicting residual component with LSTM...


['Dataset', 'Best Params', 'n_time_steps', 'MSE', 'RMSE', 'MAE', 'MAPE', 'sMAPE', 'Duration']
['sunspot', "STL(period=12)_ARIMA((0, 1, 0))_SVR({'C': 12550, 'epsilon': 0.1, 'gamma': 1e-06, 'kernel': 'rbf'})_LSTM(LSTM(8,18,8)_epochs=200)", 24, 272829.19102780544, 522.3305380961422, 484.248253564995, 75.24354024903866, 66.53, 309.4369549751282]

STL-Hybrid Results for sunspot (n_time_steps=24):
MSE: 272829.1910
RMSE: 522.3305
MAE: 484.2483
MAPE: 75.24%
sMAPE: 66.53%
Duration: 309.44s

Pipeline execution completed.
Results saved to: ../results/Result_STLASL_sunspot.csv
